# Codex Manual Strategy 20260418

Manual pipeline test copy of `strategy_dev.ipynb`.

Goal: Load data -> define strategy -> save `.py` strategy -> load strategy -> quick-test `on_event`.
            


In [1]:
import json
from pathlib import Path
import os
import numpy as np
import pandas as pd
from leadlag import list_sessions, load_session, Strategy, Order, Context, Event, BboSnapshot

DATA = '../data'
sessions = list_sessions(DATA)
assert sessions, 'No analyzed sessions found. Run collection analysis first.'
session = load_session(sessions[0]['id'], data_dir=DATA)
events = session.events.filter(signal='C')

fee_preview = ', '.join(
    f"{v}: {session.meta['fees'][v]['taker_bps']}"
    for v in session.meta['followers'][:3]
)
bbo_preview = [v for v, ok in session.meta.get('bbo_available', {}).items() if ok][:5]

print(f"Session: {session.session_id}")
print(f"Duration: {session.meta['duration_s']:.0f}s  |  Events: {session.events.count}  |  Signal C: {events.count}")
print(f"Followers: {session.meta['followers']}")
print(f"Fees (taker bps): {fee_preview}...")
print(f"BBO available: {bbo_preview}...")
            


Session: 20260417_121202_b8e21fab
Duration: 1799s  |  Events: 22  |  Signal C: 10
Followers: ['Binance Perp', 'Binance Spot', 'Bitget Perp', 'Bybit Spot', 'Gate Perp', 'Hyperliquid Perp', 'Lighter Perp', 'MEXC Perp', 'edgeX Perp']
Fees (taker bps): Binance Perp: 4.5, Binance Spot: 10.0, Bitget Perp: 6.0...
BBO available: ['OKX Perp', 'Bybit Perp', 'Binance Perp', 'Binance Spot', 'Bitget Perp']...


In [2]:
def get_notebook_strategy_name(default='codex_manual_strategy_20260418'):
    # nbconvert-friendly override; Jupyter UI can still use the runtime-session fallback.
    env_name = os.environ.get('LEADLAG_STRATEGY_NAME')
    if env_name:
        return Path(env_name).stem

    runtime_dir = Path(os.environ.get('JUPYTER_RUNTIME_DIR', '~/.jupyter/runtime')).expanduser()
    for session_file in runtime_dir.glob('kernel-*.json'):
        try:
            session_data = json.loads(session_file.read_text())
            jupyter_session = session_data.get('jupyter_session', '')
            if jupyter_session and '.ipynb' in jupyter_session:
                return Path(jupyter_session).stem
        except Exception:
            pass
    return default

STRATEGY_NAME = get_notebook_strategy_name()
print(f"Strategy name: {STRATEGY_NAME}")
            


Strategy name: codex_manual_strategy_20260418


In [3]:
from leadlag.strategy_loader import save_strategy_source

strategy_source = f"""
from leadlag import Strategy, Order


class MyStrategy(Strategy):
    name = "{STRATEGY_NAME}"
    version = "2026-04-18"
    description = "Codex manual pipeline test strategy: Signal C, Lighter follower, market entry, 30s hold."
    params = {{
        "signal": "C",
        "follower": "Lighter Perp",
        "min_magnitude": 2.0,
        "hold_ms": 30000,
    }}

    def on_event(self, event, ctx):
        p = self.params

        if event.signal != p["signal"]:
            return None
        if p["follower"] not in event.lagging_followers:
            return None
        if event.magnitude_sigma < p["min_magnitude"]:
            return None

        return Order(
            venue=p["follower"],
            side="buy" if event.direction > 0 else "sell",
            qty_btc=0.001,
            entry_type="market",
            hold_ms=p["hold_ms"],
        )
"""

strategy_path = Path('../data/strategies') / f'{STRATEGY_NAME}.py'
save_strategy_source(strategy_source, strategy_path)
print(f"Saved strategy: {strategy_path}")
            


Saved strategy: ../data/strategies/codex_manual_strategy_20260418.py


In [4]:
from leadlag import load_strategy

strat = load_strategy(strategy_path)
print(f"Loaded: {strat.name}")
print(f"Params: {strat.params}")

# Quick test with mock event
event = Event(
    bin_idx=0, ts_ms=0, signal='C', direction=1,
    magnitude_sigma=3.0, leader='OKX Perp', lagging_followers=['Lighter Perp']
)
ctx = Context(ts_ms=0, bbo={'Lighter Perp': BboSnapshot('Lighter Perp', spread_bps=1.5)})
order = strat.on_event(event, ctx)
print(f"\nTest order: {order}")
assert order is not None, 'Expected strategy to create an order for the mock Signal C event.'
            


Loaded: codex_manual_strategy_20260418
Params: {'signal': 'C', 'follower': 'Lighter Perp', 'min_magnitude': 2.0, 'hold_ms': 30000}

Test order: Order(venue='Lighter Perp', side='buy', entry_type='market', limit_price=None, hold_ms=30000, delay_ms=0, stop_loss_bps=None, take_profit_bps=None, qty_btc=0.001, tag=None)


In [5]:
# Test with params variations
print('Test 1: Change min_magnitude')
strat.params['min_magnitude'] = 5.0
order = strat.on_event(event, ctx)
print(f'  Order (stricter): {order}')
assert order is None

print('\nTest 2: Change follower')
strat.params['min_magnitude'] = 2.0
strat.params['follower'] = 'Binance Perp'
event.lagging_followers = ['Binance Perp']
order = strat.on_event(event, ctx)
print(f'  Order (other follower): {order}')
assert order is not None

# Reset
strat.params['follower'] = 'Lighter Perp'
event.lagging_followers = ['Lighter Perp']
            


Test 1: Change min_magnitude
  Order (stricter): None

Test 2: Change follower
  Order (other follower): Order(venue='Binance Perp', side='buy', entry_type='market', limit_price=None, hold_ms=30000, delay_ms=0, stop_loss_bps=None, take_profit_bps=None, qty_btc=0.001, tag=None)
